In [ ]:
# Setup. Everything below runs the stage-06b CLIs themselves, so the notebook and the
# PBS jobs execute identical code.
import logging
import os
import sys

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import pandas as pd  # noqa: E402
from IPython.display import Image, Markdown, display  # noqa: E402

from scripts import run_corrca_condition_comparison as separate_script  # noqa: E402
from scripts import run_corrca_condition_tracks as tracks_script  # noqa: E402
from scripts import run_corrca_individual_conditions as individual_script  # noqa: E402
from scripts import run_corrca_snr_grid as grid_script  # noqa: E402
from scripts.run_corrca_comparison import _build_arg_parser, run  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True
)
print("Setup complete.")

# Stage 06b — comparing CorrCA fits across bands: broadband, 1-30 Hz, 30-50 Hz

One run (`RUN`) at one `N_PCA`, fitted on the whole grid and on each band cut BEFORE the fit: does the topography chosen below 30 Hz, or on gamma alone, differ from the broadband one, and which carries the condition difference? A 1-30 Hz fit has no 40 Hz bin of its own: its SNR rows are its filters read on the broadband input (the band-swap grid), marked in the figures.

**The method is documented in
[`docs/analyses/06b-corrca-condition-comparison/`](../../docs/analyses/06b-corrca-condition-comparison/README.md)**
(section *Comparing fits*) — this notebook does not restate it. CC 1-3 of every fit are
compared by index: their ISC (the fit's own, per condition on the whole recording and on
the onset-locked maps), the held-out ISC and topography |r| of the
leave-one-participant-out refits, the Placebo/Psilocybin p-values (LOO-ISC whole and
onset, SNR contrast, the `R_w` / `R_b` terms), the SNR courses, the topographies, and
the two terms of each ISC per condition and as they move when a participant is left
out.

**What this notebook is for.** Checking the comparison cheaply before submitting
[`run_comparison.pbs`](../../jobs/metacentrum/06b-corrca-condition-comparison/run_comparison.pbs)
(its `COMPARE=band` step, one call per run of `RUNS`). The comparison is read-only, so it first needs small fits to compare: the
notebook fits each one on a subset (6 participants, 32 channels, 12 s), runs the SNR
grid on it, then calls `run()` from
[`scripts/run_corrca_comparison.py`](../../scripts/run_corrca_comparison.py) — the same
function the job calls. All three steps are the jobs' own entry points. Everything
lands under `out/comparison/`, never in `data/processed` or `plots/`, and the three
comparison notebooks share its store, so a fit one of them made is read back by the
others. The numbers mean nothing at this scale; the run checks that the tables and
figures come out as expected.

The sibling notebooks compare across
[`n_pca`](corrca_comparison_pca.ipynb) and [methods](corrca_comparison_join.ipynb).

## Configuration

In [ ]:
RUN = "tracks"  # tracks, separate, placebo_fit, psilocybin_fit or individual
N_PCA = 5  # the jobs' default
# None = broadband; the others are cut BEFORE the fit (the *_1_30hz / *_gamma jobs).
BAND_RANGES = [None, (1, 30), (30, 50)]
ENTRIES = [
    "broadband" if b is None else f"{b[0]:g}-{b[1]:g}hz" for b in BAND_RANGES
]  # the comparison's x axis, and its --bands

COMPONENTS = 3  # CC 1..COMPONENTS compared (at most 5)

# The subset every small fit runs on, as in corrca_channel_joined_separate.ipynb.
N_PAIRS = 6  # participants
N_CHANNELS = 32
N_TIMES = 3000  # samples (~12 s at 250 Hz; per condition in the time-axis join)

# One SNR-grid cell, as in corrca_component_analysis_joined_separate.ipynb.
SNR_SELECTION = "40hz"
SNR_WINDOWS = ["0-0.5s"]

EXPERIMENT = "assr"
# Shared by the three comparison notebooks, so a fit one made is read back by the others.
OUT_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06b-corrca-condition-comparison"
    / "out"
    / "comparison"
)
STORE_DIR = OUT_DIR / "store"  # the small fits' stores, never data/processed
PLOTS_DIR = OUT_DIR / "plots"  # the fits' figures and CSVs, and the comparison's
RESULTS_DIR = OUT_DIR / "results"  # the SNR grids' tables and courses
FORCE_FIT = False  # True refits (e.g. after changing the fit code)

## The fits to compare

Each fit: full montage on raw power only (the default, and what the comparison reads),
stored with its influence table, then the SNR grid on its store (for a band without
40 Hz, its filters read on the broadband store, as run_band_swap.pbs does). A fit whose store already exists is read back (`--skip_stored`),
so a rerun only redraws; set `FORCE_FIT = True` after changing the fit code.

In [ ]:
# run token -> (fit script, its extra arguments, the grid's --variant): the jobs' calls.
RUN_SCRIPTS = {
    "separate": (separate_script, [], "corrca_channel_joined_separate"),
    "placebo_fit": (
        separate_script,
        ["--fit_condition", "Placebo"],
        "corrca_channel_placebo_fit",
    ),
    "psilocybin_fit": (
        separate_script,
        ["--fit_condition", "Psilocybin"],
        "corrca_channel_psilocybin_fit",
    ),
    "tracks": (
        tracks_script,
        ["--zscore_mode", "per_condition"],
        "corrca_channel_joined_tracks",
    ),
    "individual": (individual_script, [], "corrca_channel_individual_conditions"),
}


def small_fit(run_name, n_pca, band_range=None):
    """Fit one run on the subset and store it, then run its SNR grid."""
    script, extra, variant = RUN_SCRIPTS[run_name]
    band = None if band_range is None else f"{band_range[0]:g}-{band_range[1]:g}hz"
    script.run(
        script._build_arg_parser().parse_args(
            [
                "--experiment",
                EXPERIMENT,
                "--n_pca",
                str(n_pca),
                *(["--band_range", *map(str, band_range)] if band_range else []),
                "--n_pairs",
                str(N_PAIRS),
                "--n_channels",
                str(N_CHANNELS),
                "--n_times",
                str(N_TIMES),
                "--reuse_wavelets",
                "--lenient_mask",  # a 32-channel subset drops most ASSR electrodes
                "--store_components",
                "--store_dir",
                str(STORE_DIR),
                *([] if FORCE_FIT else ["--skip_stored"]),
                "--save_dir",
                str(PLOTS_DIR),
                *extra,
            ]
        )
    )
    # A band without the 40 Hz bin (1-30 Hz) has no grid of its own: the grid reads
    # its filters on the broadband input instead (--swap_band), as run_band_swap.pbs
    # does, so the broadband fit must be stored first.
    swap = band_range is not None and not band_range[0] <= 40 <= band_range[1]
    if swap:
        print(f"{band}: no 40 Hz bin; the SNR grid reads its filters on broadband.")
    grid_script.run(
        grid_script._build_arg_parser().parse_args(
            [
                "--experiment",
                EXPERIMENT,
                "--variant",
                variant,
                "--n_pca",
                str(n_pca),
                *(["--swap_band" if swap else "--band", band] if band else []),
                "--store_dir",
                str(STORE_DIR),
                "--n_times",
                str(N_TIMES),
                "--lenient_mask",
                "--halfwidths",
                "0",
                "--stimulus_intervals",
                "0:0.5",
                "--skip_plots",
                "--save_dir",
                str(PLOTS_DIR),
                "--results_dir",
                str(RESULTS_DIR),
            ]
        )
    )


for band_range in BAND_RANGES:
    small_fit(RUN, N_PCA, band_range)

## Run the comparison

The same function the PBS job calls. The log lists every fit read (a missing store, CSV
or grid is a warning, not an error), the `R_w` / `R_b` terms per condition, and a
summary line per entry and component.

In [ ]:
args = _build_arg_parser().parse_args(
    [
        "--compare",
        "band",
        "--run",
        RUN,
        "--bands",
        *ENTRIES,
        "--n_pca",
        str(N_PCA),
        "--components",
        str(COMPONENTS),
        "--experiment",
        EXPERIMENT,
        "--snr_selection",
        SNR_SELECTION,
        "--snr_windows",
        *SNR_WINDOWS,
        "--store_dir",
        str(STORE_DIR),
        "--fit_plots_dir",
        str(PLOTS_DIR),
        "--results_dir",
        str(RESULTS_DIR),
        "--save_dir",
        str(PLOTS_DIR),
    ]
)
run(args)

## Tables and figures

`corrca_comparison_metrics.csv` is the long table behind the summary figure (one row per
fit, component, condition and metric); the `variance_*` tables hold the per-participant
`R_w` / `R_b` terms, their paired tests and their leave-one-out shift. Every figure is
drawn by [`src/visualization/corrca_plots.py`](../../src/visualization/corrca_plots.py),
which the CLI uses too.

In [ ]:
comparison_dir = (
    OUT_DIR / "plots" / "06b-corrca-condition-comparison" / "Comparison_ASSR"
)
csv = next(
    comparison_dir.glob(
        f"band__{RUN_SCRIPTS[RUN][2]}/pca_{N_PCA}/*corrca_comparison_metrics.csv"
    )
)
metrics = pd.read_csv(csv, keep_default_na=False, na_values=["nan", "NaN"])
display(
    metrics.pivot_table(
        index=["metric", "condition", "component"], columns="entry", values="value"
    ).reindex(columns=ENTRIES)
)
for png in sorted(csv.parent.glob("*corrca_comparison_*.png")):
    display(Markdown(f"**{png.relative_to(comparison_dir)}**"))
    display(Image(filename=str(png)))